In [1]:
# ============================================================
# Cell 0: GPU 可用性检查
# ============================================================
# 功能：使用 nvidia-smi 命令检查当前环境的 GPU 信息
# 输出：显示 GPU 型号、驱动版本、CUDA 版本、显存使用情况等
# 用途：确认 Colab 是否分配了 GPU，以及 GPU 的当前状态
# ============================================================
!nvidia-smi

Tue Mar  3 15:54:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ============================================================
# Cell 1: 从 GitHub 克隆项目仓库
# ============================================================
# 功能：使用 GitHub Personal Access Token (PAT) 克隆私有仓库
# 步骤：
#   1. 提示用户输入 GitHub Token (使用 getpass 隐藏输入)
#   2. 使用 Token 克隆仓库到 /content/ProjectExperiment
#   3. 检查克隆是否成功
#   4. 切换到项目目录并列出文件
#
# 注意：Token 需要具有 repo 权限才能访问私有仓库
# ============================================================
from getpass import getpass
import os

# 提示输入 GitHub Token (用于访问私有仓库)
token = getpass('Enter your GitHub Personal Access Token: ')

# 使用 Token 进行克隆
user_repo = 'szaaaaaa/ProjectExperiment'
!git clone https://{token}@github.com/{user_repo}.git

# 检查克隆是否成功并切换目录
if os.path.exists('ProjectExperiment'):
    %cd ProjectExperiment
    !ls
else:
    print("Clone failed. Please check your token or repository URL.")

### 🛠️ 如何生成 GitHub Personal Access Token (PAT)

为了访问私有仓库或通过脚本操作 GitHub，您需要一个 Token。请按照以下步骤生成：

1.  **登录 GitHub**: 访问 [github.com](https://github.com) 并登录您的账号。
2.  **进入设置**: 点击页面右上角的头像，选择 **Settings** (设置)。
3.  **开发者设置**: 在左侧边栏的最底部，点击 **Developer settings**。
4.  **选择 Token 类型**: 在左侧菜单中点击 **Personal access tokens** -> **Tokens (classic)** (推荐使用 Classic 模式，兼容性更好)。
5.  **生成新 Token**: 点击右侧的 **Generate new token** 按钮，选择 **Generate new token (classic)**。
6.  **配置 Token**:
    *   **Note**: 给 Token 起个名字，例如 `Colab Access`。
    *   **Expiration**: 设置过期时间（例如 30 days，或者 No expiration）。
    *   **Scopes (权限)**: **务必勾选 `repo`** 复选框（这会授予对私有仓库的完全读写权限）。
7.  **生成并复制**: 点击页面底部的 **Generate token** 绿涩按钮。
8.  **保存 Token**: 页面会显示一串以 `ghp_` 开头的字符，**立即复制它**（刷新页面后它将不再显示）。

👉 **接下来**: 回到 Colab，运行上面的代码单元格，当提示输入 Token 时，粘贴您刚才复制的字符串并回车。

In [ ]:
# ============================================================
# Cell 3: 挂载 Google Drive
# ============================================================
# 功能：将 Google Drive 挂载到 Colab 环境的 /content/drive 目录
# 用途：
#   - 访问存储在 Google Drive 中的数据集
#   - 保存训练结果和模型检查点到云端
#   - 实现数据和结果的持久化存储
#
# 首次运行会弹出授权窗口，需要登录 Google 账号授权
# ============================================================
from google.colab import drive

# 挂载 Google Drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# Cell 3.5: 一次性目录整理（只需运行一次）
# ============================================================
# 将视频目录从 AmuCS_experiment 移到 AmuCS 原始数据集中
# 287GB，在 Colab 挂载的 Drive 上是元数据操作（秒级完成）
#
# 运行前确认：AmuCS_experiment 目录已存在
# 运行后结果：gameplay_videos_nospeech 和 data 并列在 researchdata/ 下
# ============================================================
import os, shutil

src = "/content/drive/MyDrive/AmuCS_experiment/AmuCSVideo_unzipped/researchdata/gameplay_videos_nospeech"
dst = "/content/drive/MyDrive/AmuCS/Affective Multimodal Counter-Strike video game dataset (AMuCS) - Public/researchdata/gameplay_videos_nospeech"

if os.path.exists(src) and not os.path.exists(dst):
    print(f"Moving {src} -> {dst}")
    shutil.move(src, dst)
    print("Done!")
elif os.path.exists(dst):
    print("Already moved. Skipping.")
else:
    print(f"Source not found: {src}")

In [ ]:
# ============================================================
# Cell 4: 读取并显示基础配置文件
# ============================================================
# 功能：读取项目的基础配置文件 base.yaml 并打印其内容
# 配置文件包含：
#   - data: 数据集路径、模态列表、窗口长度等
#   - model: 模型架构参数（编码器、融合、预测头）
#   - train: 训练超参数（学习率、批大小、早停策略）
#   - eval: 评估指标
#
# 用途：了解项目的默认配置，实验配置会继承并覆盖这些默认值
# ============================================================
import yaml

config_path = 'configs/base.yaml'

# 读取配置文件内容
with open(config_path, 'r') as file:
    config_content = file.read()

print(config_content)

In [ ]:
# ============================================================
# Cell 5: 视频特征提取脚本帮助信息和目录检查
# ============================================================
# 检查 extract_video_features.py 脚本用法和视频目录内容
# ============================================================

!python scripts/extract_video_features.py --help

# 注意：视频目录需要先在 Colab 上执行 Cell 3.5 的移动命令
video_path = '/content/drive/MyDrive/AmuCS/Affective Multimodal Counter-Strike video game dataset (AMuCS) - Public/researchdata/gameplay_videos_nospeech'
!ls "{video_path}" | head -n 5

In [ ]:
# ============================================================
# Cell 6: 同步远程仓库最新代码
# ============================================================
# 功能：
#   1. 确保当前在项目根目录
#   2. 从 GitHub 拉取最新代码
#
# 用途：获取团队其他成员的最新代码更新或修复
# 注意：如果有本地未提交的修改，可能会产生冲突
# ============================================================

# 确保位于项目目录
%cd /content/ProjectExperiment

# 拉取远程仓库的最新更改
!git pull

In [ ]:
# ============================================================
# Cell 7: 游戏视频特征提取（使用 ResNet-50）
# ============================================================
# 使用预训练 ResNet-50 从游戏视频中提取帧级深度特征
# 输出：每个视频生成 .pt 文件（如 S001_P1.pt）
# ============================================================
%%bash
python scripts/extract_video_features.py \
  --video_dir "/content/drive/MyDrive/AmuCS/Affective Multimodal Counter-Strike video game dataset (AMuCS) - Public/researchdata/gameplay_videos_nospeech" \
  --output_dir "/content/drive/MyDrive/AmuCS_experiment/features/video" \
  --session_mode subdirs \
  --device cuda \
  --amp


In [ ]:
# ============================================================
# Cell 8: 配置键盘鼠标特征提取的路径
# ============================================================
ROOT_DIR = "/content/drive/MyDrive/AmuCS/Affective Multimodal Counter-Strike video game dataset (AMuCS) - Public/researchdata/data"
OUT_DIR  = "/content/drive/MyDrive/AmuCS_experiment/features/km"


In [ ]:
# ============================================================
# Cell 9: 键盘鼠标特征提取
# ============================================================
# 从原始 CSV 中提取键盘和鼠标交互行为的统计特征
# ============================================================
import os
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Source: {ROOT_DIR}")
print(f"Target: {OUT_DIR}")

!python scripts/extract_km_features.py \
  --root_dir "{ROOT_DIR}" \
  --output_dir "{OUT_DIR}" \
  --encoder stat

In [ ]:
# ============================================================
# Cell 9.5: 游戏遥测（Telemetry）特征提取
# ============================================================
# 从 gameFlt.csv + gameInt.csv 提取对齐的遥测特征
# 输出目录：/content/drive/MyDrive/AmuCS_experiment/features/telem
# ============================================================
import os

OUT_DIR_TELEM = "/content/drive/MyDrive/AmuCS_experiment/features/telem"
os.makedirs(OUT_DIR_TELEM, exist_ok=True)

print(f"Source: {ROOT_DIR}")
print(f"Target: {OUT_DIR_TELEM}")

!python scripts/extract_game_telem_features.py \
  --root_dir "{ROOT_DIR}" \
  --output_dir "{OUT_DIR_TELEM}" \
  --km_dir "/content/drive/MyDrive/AmuCS_experiment/features/km" \
  --dt 0.2



In [ ]:
# ============================================================
# Cell 10: 定义项目全局路径变量
# ============================================================
# 集中定义所有后续步骤需要的路径变量，避免硬编码
# ============================================================

PROJECT_ROOT = "/content/ProjectExperiment"
AMUCS_RAW = "/content/drive/MyDrive/AmuCS/Affective Multimodal Counter-Strike video game dataset (AMuCS) - Public/researchdata"
AMUCS_EXP = "/content/drive/MyDrive/AmuCS_experiment"

FEATURE_ROOT = f"{AMUCS_EXP}/features"          # video/, km/, telem/
ALIGNED_ROOT = f"{AMUCS_EXP}/features/aligned"   # 时间对齐后
LABELS_DIR   = f"{AMUCS_EXP}/labels"
SPLITS_DIR   = f"{AMUCS_EXP}/splits"
RUNS_DIR     = f"{AMUCS_EXP}/runs"

In [ ]:
# ============================================================
# Cell 11: 多模态特征时间对齐（video + km + telem）
# ============================================================
# 将 video、km、telem 特征对齐到统一的 5Hz 时间网格
# ============================================================
%%bash
cd /content/ProjectExperiment
python scripts/sync_data.py \
  --input_root "/content/drive/MyDrive/AmuCS_experiment/features" \
  --output_root "/content/drive/MyDrive/AmuCS_experiment/features/aligned" \
  --modalities video,km,telem \
  --grid_mode uniform \
  --target_hz 5 \
  --reference_modality video \
  --resample nearest \
  --time_origin zero \
  --report_path "/content/drive/MyDrive/AmuCS_experiment/features/aligned/alignment_report.json"

In [ ]:
# ============================================================
# Cell 12: 验证时间对齐结果
# ============================================================
import json, glob, os
report_path = "/content/drive/MyDrive/AmuCS_experiment/features/aligned/alignment_report.json"
print(json.load(open(report_path, "r", encoding="utf-8")))
print("video files:", len(glob.glob("/content/drive/MyDrive/AmuCS_experiment/features/aligned/video/*.pt")))
print("km files:", len(glob.glob("/content/drive/MyDrive/AmuCS_experiment/features/aligned/km/*.pt")))
print("telem files:", len(glob.glob("/content/drive/MyDrive/AmuCS_experiment/features/aligned/telem/*.pt")))

In [ ]:
# ============================================================
# Cell 12: 验证时间对齐结果
# ============================================================
import json, glob, os
report_path = "/content/drive/MyDrive/AmuCS_experiment/features/aligned/alignment_report.json"
print(json.load(open(report_path, "r", encoding="utf-8")))
print("video files:", len(glob.glob("/content/drive/MyDrive/AmuCS_experiment/features/aligned/video/*.pt")))
print("km files:", len(glob.glob("/content/drive/MyDrive/AmuCS_experiment/features/aligned/km/*.pt")))

In [ ]:
# ============================================================
# Cell 13: 构建时序 Arousal 标签
# ============================================================
# 从 RankTrace CSV 构建时间对齐的 arousal 序列标签
# ============================================================
%%bash
cd /content/ProjectExperiment
python scripts/build_arousal_sequence_labels.py \
  --features_root "/content/drive/MyDrive/AmuCS_experiment/features/aligned" \
  --reference_modality video \
  --ranktrace_root "/content/drive/MyDrive/AmuCS/Affective Multimodal Counter-Strike video game dataset (AMuCS) - Public/researchdata/data" \
  --output_path "/content/drive/MyDrive/AmuCS_experiment/labels/arousal_seq.json" \
  --time_col VideoTime \
  --value_col arousal \
  --time_unit ms

In [ ]:
# ============================================================
# Cell 15: 构建多模态数据集划分（按 Session 分组）
# ============================================================
%%bash
set -e
cd /content/ProjectExperiment

python scripts/build_multimodal_split.py \
  --modality_dirs "/content/drive/MyDrive/AmuCS_experiment/features/aligned/video,/content/drive/MyDrive/AmuCS_experiment/features/aligned/km,/content/drive/MyDrive/AmuCS_experiment/features/aligned/telem" \
  --labels_path "/content/drive/MyDrive/AmuCS_experiment/labels/arousal_seq.json" \
  --output_path "/content/drive/MyDrive/AmuCS_experiment/splits/session_tvt.json" \
  --val_ratio 0.2 \
  --test_ratio 0.2 \
  --split_by_session \
  --seed 42

In [ ]:
# ============================================================
# Cell 16: 时间滞后扫描分析（Lag Sweep）
# ============================================================
# 分析预测在不同时间偏移下的性能，诊断标注延迟
# 注意：此 cell 引用旧实验的 checkpoint，按需修改 RUN_DIR
# ============================================================
%%bash
set -e
cd /content/ProjectExperiment

RUN_DIR="/content/drive/MyDrive/AmuCS_experiment/runs/archive/final_lft_all_modalities_3seed/dual_video_km/2026-02-26_00-16-14__amucs_seq__lft__km_video__seed0"

python scripts/lag_sweep_seq.py \
  --config "$RUN_DIR/config.yaml" \
  --ckpt "$RUN_DIR/ckpt_best.pt" \
  --split val \
  --max_lag_sec 2 \
  --step_sec 0.2


In [ ]:
# ============================================================
# Cell 17: 序列数据管道诊断
# ============================================================
%%bash
set -e
cd /content/ProjectExperiment

python scripts/diagnose_seq_pipeline.py \
  --features_root "/content/drive/MyDrive/AmuCS_experiment/features/aligned" \
  --labels_seq_path "/content/drive/MyDrive/AmuCS_experiment/labels/arousal_seq.json" \
  --split_path "/content/drive/MyDrive/AmuCS_experiment/splits/session_tvt.json" \
  --modalities "video,km" \
  --reference_modality "video" \
  --seq_len 600 \
  --train_stride 300 \
  --val_stride 300 \
  --test_stride 300 \
  --include_tail_window \
  --sample_windows 5

In [ ]:
# ============================================================
# Cell 18: 标签归一化（Per-Participant Z-score）
# ============================================================
# 依据 AMuCS 论文：
#   "The input and target modalities were normalized
#    per participant (z-score)."
# ============================================================
%%bash
set -e
cd /content/ProjectExperiment

python scripts/normalize_seq_labels.py \
  --labels_seq_path "/content/drive/MyDrive/AmuCS_experiment/labels/arousal_seq.json" \
  --output_path "/content/drive/MyDrive/AmuCS_experiment/labels/arousal_seq_z_perparticipant.json" \
  --stats_output_path "/content/drive/MyDrive/AmuCS_experiment/labels/arousal_seq_z_perparticipant_stats.json" \
  --use_mask

In [ ]:
# ============================================================
# Cell 19: 基线模型评估（训练集均值预测）
# ============================================================
import json
from pathlib import Path
import numpy as np
import torch

features_root = Path("/content/drive/MyDrive/AmuCS_experiment/features/aligned")
labels_path = Path("/content/drive/MyDrive/AmuCS_experiment/labels/arousal_seq_z_perparticipant.json")
split_path = Path("/content/drive/MyDrive/AmuCS_experiment/splits/session_tvt.json")

modalities = ["video", "km"]
seq_len = 600
strides = {"train": 300, "val": 300, "test": 300}
include_tail = True

labels = json.loads(labels_path.read_text(encoding="utf-8-sig"))
split = json.loads(split_path.read_text(encoding="utf-8-sig"))

def load_feat_len(stem, mod):
    obj = torch.load(features_root/mod/f"{stem}.pt", map_location="cpu", weights_only=False)
    x = obj["features"] if isinstance(obj, dict) else obj
    return int(x.shape[0])

def starts_for(base_len, seq_len, stride, include_tail=True):
    if stride is None or stride <= 0:
        return [(base_len - seq_len)//2 if base_len >= seq_len else 0]
    if base_len <= seq_len:
        return [0]
    max_start = base_len - seq_len
    s = list(range(0, max_start + 1, stride))
    if include_tail and s[-1] != max_start:
        s.append(max_start)
    return s

def collect_split_values(split_name):
    ys = []
    ms = []
    for stem in split.get(split_name, []):
        if stem not in labels:
            continue
        ok = all((features_root/m/f"{stem}.pt").exists() for m in modalities)
        if not ok:
            continue
        y = np.asarray(labels[stem]["values"], dtype=np.float64)
        m = np.asarray(labels[stem].get("mask", [True]*len(y)), dtype=bool)
        base_len = min([len(y)] + [load_feat_len(stem, mm) for mm in modalities])
        y = y[:base_len]
        m = m[:base_len]
        for st in starts_for(base_len, seq_len, strides[split_name], include_tail):
            ed = min(st + seq_len, base_len)
            yw = y[st:ed]
            mw = m[st:ed]
            ys.append(yw)
            ms.append(mw)
    if not ys:
        return np.array([]), np.array([], dtype=bool)
    return np.concatenate(ys), np.concatenate(ms)

def rmse(a, b):
    return float(np.sqrt(np.mean((a-b)**2)))

def ccc(x, y):
    x = x.astype(np.float64); y = y.astype(np.float64)
    mx, my = x.mean(), y.mean()
    vx, vy = x.var(), y.var()
    cov = np.mean((x-mx)*(y-my))
    den = vx + vy + (mx-my)**2
    return float(0.0 if den == 0 else (2*cov/den))

y_train, m_train = collect_split_values("train")
train_mean = y_train[m_train].mean()
print("train_mean:", train_mean)

for sp in ["val", "test"]:
    y, m = collect_split_values(sp)
    yt = y[m]
    pred = np.full_like(yt, train_mean)
    print(f"{sp}: n={len(yt)} mean_baseline_rmse={rmse(pred, yt):.6f} ccc={ccc(pred, yt):.6f}")

In [ ]:
# ============================================================
# Cell 21: Multi-seed multi-config batch training
# ============================================================
# 7 configs x 3 seeds = 21 runs
# Labels: per-participant z-score (aligned with AMuCS paper)
# Matrix: 3 single-modality + 3 dual-modality fusion + 1 tri-modality fusion
# ============================================================
%%bash
set -euo pipefail
cd /content/ProjectExperiment

DATA_ROOT="/content/drive/MyDrive/AmuCS_experiment/features/aligned"
LABELS="/content/drive/MyDrive/AmuCS_experiment/labels/arousal_seq_z_perparticipant.json"
SPLIT="/content/drive/MyDrive/AmuCS_experiment/splits/session_tvt.json"
ROOT_RUNS="/content/drive/MyDrive/AmuCS_experiment/runs/perparticipant_z_3seed"

mkdir -p "$ROOT_RUNS"
RESULTS_TSV="$ROOT_RUNS/results.tsv"
echo -e "exp\tseed\trun_dir\tbest_val_rmse\tval_ccc\ttest_rmse\ttest_ccc" > "$RESULTS_TSV"

run_one () {
  EXP_NAME="$1"
  CFG="$2"
  SEED="$3"
  EXP_RUNS="$ROOT_RUNS/$EXP_NAME"
  mkdir -p "$EXP_RUNS"

  TS=$(date +%Y%m%d_%H%M%S)
  LOG="$EXP_RUNS/train_seed${SEED}_${TS}.log"

  echo "===== Start exp=${EXP_NAME} seed=${SEED} =====" | tee -a "$LOG"
  python -u scripts/train.py \
    --config "$CFG" \
    --override \
      data.data_root="$DATA_ROOT" \
      data.labels_seq_path="$LABELS" \
      data.split_path="$SPLIT" \
      data.seq_len=600 \
      data.train_stride=300 \
      data.val_stride=300 \
      data.test_stride=300 \
      data.num_workers=8 \
      train.batch_size=32 \
      train.optimizer.lr=2e-5 \
      train.epochs=40 \
      train.early_stopping.metric=val_rmse \
      train.early_stopping.mode=min \
      train.early_stopping.patience=5 \
      train.seed="$SEED" \
      runs_dir="$EXP_RUNS" \
    2>&1 | tee -a "$LOG"

  RUN_DIR=$(ls -dt "$EXP_RUNS"/*seed${SEED}/ | head -n1)
  METRICS_JSON="${RUN_DIR}metrics.json"

  python - <<PY
import json, pathlib
m = json.loads(pathlib.Path(r"$METRICS_JSON").read_text(encoding="utf-8"))
best_val_rmse = m.get("best_val_metric", float("nan"))
val_ccc = m.get("val_ccc", float("nan"))
test_rmse = m.get("test_rmse", float("nan"))
test_ccc = m.get("test_ccc", float("nan"))
print(f"exp=$EXP_NAME seed=$SEED run={r'$RUN_DIR'}")
print(f"best_val_rmse={best_val_rmse} val_ccc={val_ccc} test_rmse={test_rmse} test_ccc={test_ccc}")
with open(r"$RESULTS_TSV", "a", encoding="utf-8") as f:
    f.write(f"$EXP_NAME\\t$SEED\\t{r'$RUN_DIR'}\\t{best_val_rmse}\\t{val_ccc}\\t{test_rmse}\\t{test_ccc}\\n")
PY
}

# --- Single-modality ---
for SEED in 0 1 2; do
  run_one "single_video" "configs/experiments/video_lft_aligned_seq.yaml" "$SEED"
done

for SEED in 0 1 2; do
  run_one "single_km" "configs/experiments/km_lft_aligned_seq.yaml" "$SEED"
done

for SEED in 0 1 2; do
  run_one "single_telem" "configs/experiments/telem_lft_aligned_seq.yaml" "$SEED"
done

# --- Dual-modality ---
for SEED in 0 1 2; do
  run_one "dual_video_km" "configs/experiments/video_km_aligned_seq.yaml" "$SEED"
done

for SEED in 0 1 2; do
  run_one "dual_video_telem" "configs/experiments/video_telem_lft_aligned_seq.yaml" "$SEED"
done

for SEED in 0 1 2; do
  run_one "dual_km_telem" "configs/experiments/km_telem_lft_aligned_seq.yaml" "$SEED"
done

# --- Tri-modality ---
for SEED in 0 1 2; do
  run_one "triple_video_km_telem" "configs/experiments/video_km_telem_aligned_seq.yaml" "$SEED"
done

python - <<PY
import pandas as pd
from pathlib import Path
p = Path(r"$RESULTS_TSV")
df = pd.read_csv(p, sep="\t")
print("\n=== Per-run Results ===")
print(df[["exp","seed","best_val_rmse","val_ccc","test_rmse","test_ccc"]].to_string(index=False))

g = df.groupby("exp")[["best_val_rmse","val_ccc","test_rmse","test_ccc"]].agg(["mean","std"])
print("\n=== 3-seed Summary (mean/std) ===")
print(g)

out_csv = p.with_name("results_summary.csv")
g.to_csv(out_csv)
print(f"\nSaved:\n- {p}\n- {out_csv}")
PY


